# Homework 10 — Spark Structured Streaming
Michelle Silveira

This notebook covers:
- **Part 1**: Creating a streaming data source with `rate`, applying transformations, writing to memory.
- **Part 2**: Fitting a pipeline on a static CSV, then using it to transform streaming CSV files dropped into a folder.

Works on both **JupyterHub** (PySpark pre-installed) and **Google Colab** (installs PySpark automatically).

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('HW10_Streaming') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')  # suppress INFO noise
print('Spark version:', spark.version)
print('Session ready\!')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 16:31:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/21 16:31:04 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/21 16:31:04 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/21 16:31:04 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark version: 4.0.1
Session ready\!


---
# Part 1 — Creating Streaming Data Using `rate`

The `rate` source generates rows automatically at a steady pace.  
Each row has two columns: `timestamp` and `value` (an incrementing integer).

We will add two derived columns:
- `sqrt_value` — square root of `value`
- `mod4_value` — `value mod 4`

In [2]:
from pyspark.sql.functions import col, sqrt

# Create the streaming source
rate_stream = spark.readStream.format('rate').load()

# Apply transformations
rate_transformed = rate_stream.select(
    col('timestamp'),
    col('value'),
    sqrt(col('value')).alias('sqrt_value'),
    (col('value') % 4).alias('mod4_value')
)

print('Stream schema:')
rate_transformed.printSchema()

Stream schema:
root
 |-- timestamp: timestamp (nullable = true)
 |-- value: long (nullable = true)
 |-- sqrt_value: double (nullable = true)
 |-- mod4_value: long (nullable = true)



We write the stream to an **in-memory table** called `rate_data`.  
The query starts running in the background immediately.

In [3]:
query = rate_transformed.writeStream \
    .format('memory') \
    .queryName('rate_data') \
    .start()

print('Query started\! Name:', query.name)
print('Is active:', query.isActive)

26/04/21 16:32:52 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-193e36ac-5b0f-4865-b971-e412fc371427. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:32:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Query started\! Name: rate_data
Is active: True


We wait 30 seconds while the stream collects rows into the in-memory table.  
A progress counter prints every 5 seconds so you can confirm it's working.

In [4]:
import time

print('Waiting 30 seconds for data to accumulate...')
for i in range(6):
    time.sleep(5)
    count = spark.sql('SELECT count(*) as n FROM rate_data').collect()[0]['n']
    print(f'  {5*(i+1)}s elapsed — rows so far: {count}')
print('Done waiting\!')

Waiting 30 seconds for data to accumulate...
  5s elapsed — rows so far: 46
  10s elapsed — rows so far: 52
  15s elapsed — rows so far: 57
  20s elapsed — rows so far: 62
  25s elapsed — rows so far: 67
  30s elapsed — rows so far: 72
Done waiting\!


Stop the stream and query the full in-memory table with `spark.sql()`.

In [20]:
query.stop()
print('Query stopped.')

# Show all rows collected during the 30 seconds
spark.sql('SELECT * FROM rate_data').show(200)

Query stopped.
+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-21 16:32:...|    0|               0.0|         0|
|2026-04-21 16:32:...|    1|               1.0|         1|
|2026-04-21 16:32:...|    2|1.4142135623730951|         2|
|2026-04-21 16:32:...|    3|1.7320508075688772|         3|
|2026-04-21 16:32:...|    4|               2.0|         0|
|2026-04-21 16:32:...|    5|  2.23606797749979|         1|
|2026-04-21 16:32:...|    6| 2.449489742783178|         2|
|2026-04-21 16:32:...|    7|2.6457513110645907|         3|
|2026-04-21 16:33:...|    8|2.8284271247461903|         0|
|2026-04-21 16:33:...|    9|               3.0|         1|
|2026-04-21 16:33:...|   10|3.1622776601683795|         2|
|2026-04-21 16:33:...|   11|   3.3166247903554|         3|
|2026-04-21 16:33:...|   12|3.4641016151377544|         0|
|2026-04-21 16:33:...|   13| 3.6055512754

---
# Part 2 — Using CSV Data with a Pipeline

Steps:
1. Read `bikeDetails_for_fit.csv` as a Spark SQL DataFrame.
2. Build an `SQLTransformer` + `VectorAssembler` pipeline and fit it.
3. Set up a `readStream` that watches a folder for incoming CSV files.
4. Apply the fitted pipeline's `.transform()` to each new file.
5. Write results to the console in `append` mode.

We download the file locally first so Spark reads it with a proper inferred schema on both platforms.

In [7]:
import os
import urllib.request
import pandas as pd

# ── Download the original full dataset (this URL works) ──────────────────────
urllib.request.urlretrieve(
    "https://www4.stat.ncsu.edu/~online/datasets/bikeDetails.csv",
    "bikeDetails.csv"
)
print("Downloaded bikeDetails.csv")

# ── Split it to match the homework's 6-file structure ────────────────────────
full = pd.read_csv("bikeDetails.csv").dropna(subset=["selling_price", "km_driven", "owner", "year"])
full = full.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle

# 80% → training file,  20% → 5 streaming chunks
split_idx = int(len(full) * 0.8)
fit_df    = full.iloc[:split_idx]
add_df    = full.iloc[split_idx:]

fit_df.to_csv("bikeDetails_for_fit.csv", index=False)
print(f"bikeDetails_for_fit.csv  → {len(fit_df)} rows")

# Split the remaining 20% into 5 equal chunks
chunks = [add_df.iloc[i::5] for i in range(5)]
os.makedirs("bike_staging", exist_ok=True)
for i, chunk in enumerate(chunks, start=1):
    fname = f"bike_staging/bikeDetails_add{i}.csv"
    chunk.to_csv(fname, index=False)
    print(f"bikeDetails_add{i}.csv       → {len(chunk)} rows")

# ── Read training file into Spark ─────────────────────────────────────────────
bike_fit = spark.read.csv("bikeDetails_for_fit.csv", header=True, inferSchema=True)
print(f"\nSpark DataFrame ready — {bike_fit.count()} rows")
bike_fit.show(5)
print("Schema:", bike_fit.schema)

Downloaded bikeDetails.csv
bikeDetails_for_fit.csv  → 848 rows
bikeDetails_add1.csv       → 43 rows
bikeDetails_add2.csv       → 43 rows
bikeDetails_add3.csv       → 43 rows
bikeDetails_add4.csv       → 42 rows
bikeDetails_add5.csv       → 42 rows

Spark DataFrame ready — 848 rows
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|                name|selling_price|year|seller_type|    owner|km_driven|ex_showroom_price|
+--------------------+-------------+----+-----------+---------+---------+-----------------+
|Yamaha FZ S [2012...|        38000|2013| Individual|1st owner|    75000|          79432.0|
|Suzuki Access 125...|        16000|2011| Individual|1st owner|    40000|          58314.0|
|Hero Honda CD Deluxe|        12000|2007| Individual|2nd owner|   100000|             NULL|
|       Suzuki GS150R|        27000|2012| Individual|1st owner|    14100|          70851.0|
|     Hero Xpulse 200|       100000|2019| Individual|1st owner|     8500| 

## Create the SQLTransformer

The exact SQL statement required by the homework:  
- Logs `selling_price` → `label`  
- Logs `km_driven` → `log_km_driven`  
- Creates `one_owner` indicator from the `owner` string column

In [8]:
from pyspark.ml.feature import SQLTransformer

sql_trans = SQLTransformer(
    statement="""
        SELECT log(selling_price) as label,
               year,
               log(km_driven) as log_km_driven,
               CASE WHEN owner = '1st owner' THEN 1 ELSE 0 END AS one_owner
        FROM __THIS__
    """
)

# Quick sanity check on the training data
sql_trans.transform(bike_fit).show(5)

+------------------+----+------------------+---------+
|             label|year|     log_km_driven|one_owner|
+------------------+----+------------------+---------+
|10.545341438708522|2013|11.225243392518447|        1|
| 9.680344001221918|2011|10.596634733096073|        1|
| 9.392661928770137|2007|11.512925464970229|        0|
|10.203592144986466|2012|  9.55393007636626|        1|
|11.512925464970229|2019| 9.047821442478408|        1|
+------------------+----+------------------+---------+
only showing top 5 rows


Create the VectorAssembler

MLlib requires all predictor columns packed into a single `features` vector column.

In [9]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=['year', 'log_km_driven', 'one_owner'],
    outputCol='features',
    handleInvalid='keep'
)

# Quick check — chain SQLTransformer then assembler
assembler.transform(
    sql_trans.transform(bike_fit)
).show(5)

+------------------+----+------------------+---------+--------------------+
|             label|year|     log_km_driven|one_owner|            features|
+------------------+----+------------------+---------+--------------------+
|10.545341438708522|2013|11.225243392518447|        1|[2013.0,11.225243...|
| 9.680344001221918|2011|10.596634733096073|        1|[2011.0,10.596634...|
| 9.392661928770137|2007|11.512925464970229|        0|[2007.0,11.512925...|
|10.203592144986466|2012|  9.55393007636626|        1|[2012.0,9.5539300...|
|11.512925464970229|2019| 9.047821442478408|        1|[2019.0,9.0478214...|
+------------------+----+------------------+---------+--------------------+
only showing top 5 rows


## Build the Pipeline and Fit It

We chain the two stages and fit the pipeline on the training DataFrame.  
The `fitted_pipeline` object can later `.transform()` any DataFrame — including streaming ones.

In [10]:
from pyspark.ml import Pipeline

pipeline = Pipeline(stages=[sql_trans, assembler])
fitted_pipeline = pipeline.fit(bike_fit)

print('Pipeline fitted\!')

# Verify on training data
fitted_pipeline.transform(bike_fit).select('label', 'features').show(5)

Pipeline fitted\!
+------------------+--------------------+
|             label|            features|
+------------------+--------------------+
|10.545341438708522|[2013.0,11.225243...|
| 9.680344001221918|[2011.0,10.596634...|
| 9.392661928770137|[2007.0,11.512925...|
|10.203592144986466|[2012.0,9.5539300...|
|11.512925464970229|[2019.0,9.0478214...|
+------------------+--------------------+
only showing top 5 rows


Set Up readStream and writeStream, Then Start

- Schema comes from `bike_fit.schema` so column types match exactly.
- `option('header', 'true')` is needed because each add CSV has a header row.
- The fitted pipeline transforms each incoming micro-batch.
- Results print to the **console** in **append** mode.



In [12]:
import shutil, os

STAGING_DIR = "bike_staging"
STREAM_DIR  = "bike_stream"

# Make sure stream folder exists and is empty
if os.path.exists(STREAM_DIR):
    shutil.rmtree(STREAM_DIR)
os.makedirs(STREAM_DIR)

print("STAGING_DIR:", STAGING_DIR)
print("STREAM_DIR:", STREAM_DIR)
print("Stream folder empty:", os.listdir(STREAM_DIR) == [])

STAGING_DIR: bike_staging
STREAM_DIR: bike_stream
Stream folder empty: True


In [13]:
bike_schema = bike_fit.schema

# Read stream — watches the folder for new CSV files
stream_input = spark.readStream \
    .schema(bike_schema) \
    .option('header', 'true') \
    .csv(STREAM_DIR)

# Transform each micro-batch with the fitted pipeline
stream_output = fitted_pipeline.transform(stream_input)

# Write to console in append mode
stream_query = stream_output.writeStream \
    .format('console') \
    .outputMode('append') \
    .option('truncate', 'false') \
    .start()

print('Stream query started\!')
print('Watching folder:', STREAM_DIR)
print('Now run the blocks below to add files one at a time.')

Stream query started\!
Watching folder: bike_stream
Now run the blocks below to add files one at a time.


26/04/21 16:41:44 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-d5b9ef1d-cb4f-4657-b8ac-adc81aa61740. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/21 16:41:44 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Add File 1

Run each of the next 5 blocks one at a time. After each, watch the console output (in Block 12's output area) for the transformed rows.

In [14]:
import time, shutil, os

fname = 'bikeDetails_add1.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done. Check console output.')

Copied bikeDetails_add1.csv -> bike_stream/
-------------------------------------------
Batch: 0
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|9.615805480084347 |2005|10.772057548198871|1        |[2005.0,10.772057548198871,1.0]|
|10.308952660644293|2012|8.517193191416238 |0        |[2012.0,8.517193191416238,0.0] |
|10.126631103850338|2015|10.596634733096073|0        |[2015.0,10.596634733096073,0.0]|
|12.230765258120545|2018|8.922658299524402 |1        |[2018.0,8.922658299524402,1.0] |
|9.740968623038354 |2010|13.122363377404328|1        |[2010.0,13.122363377404328,1.0]|
|10.308952660644293|2011|11.156250521031495|0        |[2011.0,11.156250521031495,0.0]|
|10.819778284410283|2014|7.090076835776092 |1        |[2014.0,7.090076835776

Add File 2

In [15]:
fname = 'bikeDetails_add2.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done.')

Copied bikeDetails_add2.csv -> bike_stream/
-------------------------------------------
Batch: 1
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.126631103850338|2008|10.545341438708522|1        |[2008.0,10.545341438708522,1.0]|
|10.126631103850338|2011|9.903487552536127 |1        |[2011.0,9.903487552536127,1.0] |
|11.77528972943772 |2017|8.294049640102028 |1        |[2017.0,8.294049640102028,1.0] |
|10.114558522616068|2012|10.596634733096073|0        |[2012.0,10.596634733096073,0.0]|
|11.695247021764184|2016|12.751299696013497|1        |[2016.0,12.751299696013497,1.0]|
|11.156250521031495|2017|9.472704636443673 |1        |[2017.0,9.472704636443673,1.0] |
|11.156250521031495|2016|9.210340371976184 |1        |[2016.0,9.210340371976

Add File 3

In [16]:
fname = 'bikeDetails_add3.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done.')

Copied bikeDetails_add3.csv -> bike_stream/
-------------------------------------------
Batch: 2
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.239959789157341|2008|11.168870552387599|1        |[2008.0,11.168870552387599,1.0]|
|10.308952660644293|2014|10.596634733096073|1        |[2014.0,10.596634733096073,1.0]|
|11.082142548877775|2011|11.082142548877775|0        |[2011.0,11.082142548877775,0.0]|
|10.819778284410283|2014|11.156250521031495|1        |[2014.0,11.156250521031495,1.0]|
|10.714417768752456|2014|9.581903928408666 |1        |[2014.0,9.581903928408666,1.0] |
|10.714417768752456|2017|10.799575577092764|1        |[2017.0,10.799575577092764,1.0]|
|9.210340371976184 |2006|11.225243392518447|1        |[2006.0,11.22524339251

Add File 4

In [17]:
fname = 'bikeDetails_add4.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done.')

Copied bikeDetails_add4.csv -> bike_stream/
-------------------------------------------
Batch: 3
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|11.512925464970229|2015|10.747207591575448|1        |[2015.0,10.747207591575448,1.0]|
|11.695247021764184|2018|8.699514748210191 |1        |[2018.0,8.699514748210191,1.0] |
|11.407564949312402|2007|8.160518247477505 |1        |[2007.0,8.160518247477505,1.0] |
|9.903487552536127 |2006|9.159047077588632 |1        |[2006.0,9.159047077588632,1.0] |
|11.225243392518447|2017|9.798127036878302 |1        |[2017.0,9.798127036878302,1.0] |
|9.903487552536127 |2005|9.615805480084347 |1        |[2005.0,9.615805480084347,1.0] |
|10.126631103850338|2009|10.819778284410283|1        |[2009.0,10.81977828441

Add File 5

In [18]:
fname = 'bikeDetails_add5.csv'
shutil.copy(os.path.join(STAGING_DIR, fname), os.path.join(STREAM_DIR, fname))
print(f'Copied {fname} -> {STREAM_DIR}/')
time.sleep(5)
print('Done. All 5 files processed\!')

Copied bikeDetails_add5.csv -> bike_stream/
-------------------------------------------
Batch: 4
-------------------------------------------
+------------------+----+------------------+---------+-------------------------------+
|label             |year|log_km_driven     |one_owner|features                       |
+------------------+----+------------------+---------+-------------------------------+
|10.596634733096073|2016|10.4341158035983  |1        |[2016.0,10.4341158035983,1.0]  |
|10.46310334047155 |2011|10.819778284410283|1        |[2011.0,10.819778284410283,1.0]|
|10.46310334047155 |2018|10.085809109330082|1        |[2018.0,10.085809109330082,1.0]|
|10.714417768752456|2013|9.903487552536127 |1        |[2013.0,9.903487552536127,1.0] |
|9.615805480084347 |2006|10.819778284410283|1        |[2006.0,10.819778284410283,1.0]|
|11.608235644774552|2017|9.375854810453756 |1        |[2017.0,9.375854810453756,1.0] |
|12.254862809699606|2017|9.392661928770137 |1        |[2017.0,9.392661928770

Stop the Stream Query

Once all files are processed and console output has appeared, stop the query.

In [19]:
stream_query.stop()
print('Stream query stopped.')
print('Active streams remaining:', spark.streams.active)

Stream query stopped.
Active streams remaining: []


26/04/21 16:45:15 WARN DAGScheduler: Failed to cancel job group 9cd558a0-fdda-4ae1-9309-1d4442f1739c. Cannot find active jobs for it.
26/04/21 16:45:15 WARN DAGScheduler: Failed to cancel job group 9cd558a0-fdda-4ae1-9309-1d4442f1739c. Cannot find active jobs for it.
